# R3maJ Kaggle Notebook v4

Kaggle GPU training for R3maJ with Google Drive replay **and checkpoint restore**.


In [ ]:
# 1. Clone R3maJ
import os, subprocess
ROOT='/kaggle/working/R3maJ'
REPO='https://github.com/vfxjamer/R3maJ.git'
if not os.path.isdir(os.path.join(ROOT,'.git')):
    subprocess.run(['git','clone','--depth','1',REPO,ROOT],check=True)
else:
    subprocess.run(['git','-C',ROOT,'pull'],check=False)
print(ROOT)


In [ ]:
# 2. Download replay + checkpoints from Google Drive
!pip install -q gdown
import gdown, os
REPLAY_FILE_ID='YOUR_REPLAY_FILE_ID'
CHECKPOINT_FOLDER_ID='YOUR_CHECKPOINT_FOLDER_ID'
LOCAL_REPLAY='/kaggle/working/R3maJ/build/serialized_replays.bin'
LOCAL_CHECKPOINTS='/kaggle/working/R3maJ/build/checkpoints'
os.makedirs(os.path.dirname(LOCAL_REPLAY),exist_ok=True)
os.makedirs(LOCAL_CHECKPOINTS,exist_ok=True)
assert REPLAY_FILE_ID!='YOUR_REPLAY_FILE_ID','Set REPLAY_FILE_ID first.'
assert CHECKPOINT_FOLDER_ID!='YOUR_CHECKPOINT_FOLDER_ID','Set CHECKPOINT_FOLDER_ID first.'
if not os.path.exists(LOCAL_REPLAY):
    gdown.download(f'https://drive.google.com/uc?id={REPLAY_FILE_ID}',LOCAL_REPLAY,quiet=False)
print('Downloading checkpoint folder...')
gdown.download_folder(f'https://drive.google.com/drive/folders/{CHECKPOINT_FOLDER_ID}',output=LOCAL_CHECKPOINTS,quiet=False,use_cookies=False)
print('Replay:',os.path.exists(LOCAL_REPLAY))
print('Replay GB:',round(os.path.getsize(LOCAL_REPLAY)/(1024**3),3) if os.path.exists(LOCAL_REPLAY) else 0)
print('Checkpoint entries:',sorted(os.listdir(LOCAL_CHECKPOINTS))[:20])


### IDs
`REPLAY_FILE_ID` = the ID in your `serialized_replays.bin` file URL.
`CHECKPOINT_FOLDER_ID` = the ID in your `checkpoints` folder URL.
Both must be accessible to the Kaggle download.


In [ ]:
# 3. Install build dependencies + inspect GPUs
import subprocess,os
subprocess.run(['apt-get','update','-qq'],check=False)
subprocess.run(['apt-get','install','-y','-qq','build-essential','cmake','git','libpython3-dev','pkg-config'],check=False)
import torch
print('torch:',torch.__version__)
print('CUDA:',torch.cuda.is_available(),torch.version.cuda)
print('GPU count:',torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(i,torch.cuda.get_device_name(i))


In [ ]:
# 4. Configure + build
import os,subprocess,torch
os.chdir(ROOT)
prefix=os.path.dirname(torch.__file__)
subprocess.run(['cmake','-S','.','-B','build','-DCMAKE_BUILD_TYPE=Release',f'-DTORCH_INSTALL_PREFIX={prefix}'],check=True)
subprocess.run(['cmake','--build','build','-j',str(os.cpu_count() or 2)],check=True)
EXE=os.path.join(ROOT,'build','R3maJ')
print('Binary:',EXE,os.path.exists(EXE))


In [ ]:
# 5. Verify restored data
assert os.path.exists(EXE),'R3maJ binary missing.'
assert os.path.exists(LOCAL_REPLAY),'Replay missing.'
assert os.path.isdir(LOCAL_CHECKPOINTS),'Checkpoint directory missing.'
print('READY')
print('checkpoint entries:',len([x for x in os.listdir(LOCAL_CHECKPOINTS) if not x.startswith('.')]))


In [ ]:
# 6. Start training
import os,subprocess
os.chdir(os.path.join(ROOT,'build'))
TRAIN_ARGS=['--device','cuda','--save-dir','checkpoints','--games','164','--replays','serialized_replays.bin']
print('Launching:','./R3maJ',*TRAIN_ARGS)
proc=subprocess.Popen(['./R3maJ']+TRAIN_ARGS)
print('Training PID:',proc.pid)


## Important
This v4 restores checkpoints from Google Drive at startup. `gdown` is download-only, so checkpoints created during this Kaggle run are **not automatically uploaded back to Drive**. Kaggle runtime storage is temporary. A separate upload/sync mechanism is required if you want every new checkpoint persisted to Drive.
